In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
Data=pd.read_csv("InsurenceData.csv",index_col=0)
Data.drop(columns=["Churn","Customer Name"],inplace=True)
Data.head()

## Optimized Machine Learning Pipeline
Fixing resources issue by dropping high-cardinality columns and setting n_jobs=1.

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Features and Target
X=Data.drop(columns=['Claim Request output','Customer_Address'])
y=Data['Claim Request output'].map({'Yes':1,'No':0})

# 2. Optimized Preprocessing (Dropping high-cardinality Company Name for stability)
categorical_features=['Claim Reason','Data confidentiality']
numerical_features=['Claim Amount','Premium/Amount Ratio']

preprocessor=ColumnTransformer([
    ('num',StandardScaler(),numerical_features),
    ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)
])

# 3. Handle Imbalance
ros=RandomOverSampler(sampling_strategy=0.4,random_state=42)
X_res,y_res=ros.fit_resample(X.drop(columns=['Company Name']),y)

X_train,X_test,y_train,y_test=train_test_split(X_res,y_res,test_size=0.2,random_state=42)

# 4. Training Suite
models_dict={
    'XGBoost':(XGBClassifier(n_jobs=1),{'n_estimators':[50],'max_depth':[3,5]}),
    'LightGBM':(LGBMClassifier(n_jobs=1),{'n_estimators':[50]}),
    'RandomForest':(RandomForestClassifier(n_jobs=1),{'n_estimators':[50]}),
    'LogisticReg':(LogisticRegression(max_iter=500),{'C':[1]})
}

results=[]
for name,(model,params) in models_dict.items():
    print(f'Training {name}...')
    pipe=Pipeline([('prep',preprocessor),('model',model)])
    pipe_params={f'model__{k}':v for k,v in params.items()}
    search=RandomizedSearchCV(pipe,pipe_params,n_iter=2,cv=3,scoring='f1',random_state=42,n_jobs=1)
    search.fit(X_train,y_train)
    y_pred=search.predict(X_test)
    results.append({
        'Model':name,
        'Best Params':search.best_params_,
        'F1 Score':f1_score(y_test,y_pred),
        'Accuracy':accuracy_score(y_test,y_pred)
    })

results_df=pd.DataFrame(results).sort_values(by='F1 Score',ascending=False)
print(results_df)

## Final Training and Save

In [7]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import RandomOverSampler

In [8]:
Data=pd.read_csv("InsurenceData.csv",index_col=0)
Data.drop(columns=["Churn","Customer Name"],inplace=True)
Data.head()

,Customer_Address,Company Name,Claim Reason,Data confidentiality,Claim Amount,Category Premium,Premium/Amount Ratio,Claim Request output,BMI
0,"7627 Anderson Rest Apt. 265,Lake Heather, DC 3...","Williams, Henderson and Perez",Travel,Low,377,4794,0.078640,No,21
1,"3953 Cindy Brook Apt. 147,East Lindatown, TN 4...",Moore-Goodwin,Medical,High,1440,14390,0.100069,No,24
2,"8693 Walters Mountains,South Tony, TX 88407",Smith-Holmes,Phone,Medium,256,1875,0.136533,No,18
3,"56926 Webster Coves,Shawnmouth, NV 04853",Harrell-Perez,Phone,Medium,233,1875,0.124267,No,24
4,"489 Thomas Forges Apt. 305,Jesseton, GA 36765","Simpson, Kramer and Hughes",Phone,Medium,239,1875,0.127467,No,21


In [9]:
# 1. Features and Target
X=Data.drop(columns=['Claim Request output','Customer_Address'])
y=Data['Claim Request output'].map({'Yes':1,'No':0})

# 2. Optimized Preprocessing (Dropping high-cardinality Company Name for stability)
categorical_features=['Claim Reason','Data confidentiality']
numerical_features=['Claim Amount','Premium/Amount Ratio']

preprocessor=ColumnTransformer([
    ('num',StandardScaler(),numerical_features),
    ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)
])

# 3. Handle Imbalance
ros=RandomOverSampler(sampling_strategy=0.4,random_state=42)
X_res,y_res=ros.fit_resample(X.drop(columns=['Company Name']),y)

In [10]:
import joblib
final_model=XGBClassifier(n_estimators=100,learning_rate=0.1,max_depth=5,n_jobs=1)
final_pipe=Pipeline([('prep',preprocessor),('model',final_model)])
final_pipe.fit(X_res,y_res)
joblib.dump(final_pipe,'final_insurance_model.pkl')
print('Model saved successfully!')

Model saved successfully!
